In [1]:
# Imports and Configurations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.regression.linear_model import OLS 
from statsmodels.tools import add_constant
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

np.random.seed(42)

In [4]:
# Laading saved outputs from Notebook 2
sharpe_data = pd.read_csv('sharpe_data.csv', index_col=0, parse_dates=True)

cluster_k2 = pd.read_csv('cluster_assignments_sharpe_k2.csv')
cluster_k4 = pd.read_csv('cluster_assignments_sharpe_k4.csv')
cluster_k7 = pd.read_csv('cluster_assignments_sharpe_k7.csv')

print('Sharpe data shape:', sharpe_data.shape)
print('Date range:', sharpe_data.index[0].date(), 'to', sharpe_data.index[-1].date())
print('\nCluster assignment shapes:')
print(f' k=2: {cluster_k2.shape}, k=4:  {cluster_k4.shape}, k=7 = {cluster_k7.shape}')
print('\nSample cluster assignments (k=4, first 5 rows):')
print(cluster_k4.head())

Sharpe data shape: (470, 77)
Date range: 2014-12-31 to 2023-12-27

Cluster assignment shapes:
 k=2: (77, 3), k=4:  (77, 3), k=7 = (77, 3)

Sample cluster assignments (k=4, first 5 rows):
  Ticker  Cluster       Sector
0   AAPL        2   Technology
1   ADBE        1   Technology
2    ADI        1   Technology
3    ADP        4  Industrials
4   ALGN        3   Healthcare


In [7]:
# Redownloading price data
import yfinance as yf

tickers = sharpe_data.columns.tolist()

print(f'Re-downloading price data for {len(tickers)} stocks...')
print('Using same configuration as Notebook 2: weekly Wednesday, 2014-2023\n')

raw = yf.download(
    tickers,
    start='2014-01-01',
    end='2024-01-01',
    interval='1wk', 
    auto_adjust=True,
    progress=True
)

price_data = raw['Close']
price_data = price_data.resample('W-WED').last()
price_data = price_data.dropna(how='all')
price_data = price_data[tickers]
price_data = price_data.ffill().dropna(axis=1)

# Keeping only tickers present in both sharpe_data and price_data
common_tickers  = [t for t in tickers if t in price_data.columns]
price_data = price_data[common_tickers]

print(f'\nPrice data shape: {price_data.shape}')
print(f'Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}')
print(f'Tickers retained: {len(common_tickers)}')

print(sharpe_data.columns.tolist() == common_tickers)

[****                   9%                       ]  7 of 77 completed

Re-downloading price data for 77 stocks...
Using same configuration as Notebook 2: weekly Wednesday, 2014-2023



[*********************100%***********************]  77 of 77 completed


Price data shape: (522, 77)
Date range: 2014-01-01 to 2023-12-27
Tickers retained: 77
True


In [10]:
# Building cluster lookup

cluster_map = {}

for k, df in [(2, cluster_k2), (4, cluster_k4), (7, cluster_k7)]:
    cluster_map[k] = {}
    for cluster_num in sorted(df['Cluster'].unique()):
        tickers_in_cluster = df[df['Cluster'] == cluster_num]['Ticker'].tolist()
        cluster_map[k][cluster_num] = tickers_in_cluster

for k in [2, 4, 7]:
    print(f'\nk={k}:')
    for cluster_num, tickers in cluster_map[k].items():
        print(f' Cluster {cluster_num} ({len(tickers)} stocks): {', '.join(tickers)}')


k=2:
 Cluster 1 (38 stocks): ADBE, ADI, AMAT, AMD, AMZN, ASML, BKNG, BKR, CCEP, CDNS, CPRT, FANG, FAST, GOOG, GOOGL, IDXX, INTU, KLAC, LIN, LRCX, MCHP, MELI, META, MRVL, MSFT, MU, NFLX, NVDA, ODFL, ON, PCAR, QCOM, SNPS, TSLA, VRSK, VRTX, WBD, WDAY
 Cluster 2 (39 stocks): AAPL, ADP, ALGN, AMGN, AVGO, AZN, BIIB, CDW, COST, CSCO, CSGP, CTAS, CTSH, DLTR, DXCM, EA, ENPH, EXC, FTNT, GILD, HON, ILMN, ISRG, KDP, MAR, MDLZ, MNST, MTCH, ORLY, PANW, PAYX, PEP, REGN, ROST, SBUX, SIRI, SMCI, TXN, XEL

k=4:
 Cluster 1 (38 stocks): ADBE, ADI, AMAT, AMD, AMZN, ASML, BKNG, BKR, CCEP, CDNS, CPRT, FANG, FAST, GOOG, GOOGL, IDXX, INTU, KLAC, LIN, LRCX, MCHP, MELI, META, MRVL, MSFT, MU, NFLX, NVDA, ODFL, ON, PCAR, QCOM, SNPS, TSLA, VRSK, VRTX, WBD, WDAY
 Cluster 2 (11 stocks): AAPL, AVGO, COST, CTSH, GILD, MAR, MNST, ORLY, PANW, REGN, SMCI
 Cluster 3 (11 stocks): ALGN, AZN, CSGP, DXCM, ENPH, HON, ISRG, MDLZ, MTCH, SBUX, SIRI
 Cluster 4 (17 stocks): ADP, AMGN, BIIB, CDW, CSCO, CTAS, DLTR, EA, EXC, FTNT, ILM